<a href="https://colab.research.google.com/github/claudiodanielpc/medium/blob/main/rezago_r_srvyr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
##Borrar datos del entorno
rm(list=ls())


#Se utiliza pacman para instalar y cargar paquetes
if(!require('pacman')) install.packages('pacman')
pacman::p_load(tidyverse,janitor,srvyr)

Loading required package: pacman



In [25]:
# -------------------------------
# Datos de viviendas ENIGH 2024
# -------------------------------
url_vivi <- "https://www.inegi.org.mx/contenidos/programas/enigh/nc/2024/microdatos/enigh2024_ns_viviendas_csv.zip"

In [26]:
# -------------------------------
# Descargar ZIP
# -------------------------------
zip_path <- "enigh2024_viviendas.zip"
download.file(
  url_vivi,
  destfile = zip_path,
  mode = "wb"
)
# -------------------------------
# Crear carpeta destino
# -------------------------------
output_dir <- "enigh2024_viviendas"
dir.create(output_dir, showWarnings = FALSE)


In [27]:
# -------------------------------
# Descomprimir
# -------------------------------
unzip(zip_path, exdir = output_dir)

# Ver archivos
list.files(output_dir)


[1] "viviendas.csv"

In [32]:
## Leer viviendas
viviendas <- read_csv(paste0(output_dir,"/viviendas.csv"))

valor_excusado <- if (length(unique(viviendas$excusado)) == 3) 3 else 2
viviendas<-viviendas%>%
  janitor::clean_names()%>%
  #Formato de variables clave para rezago
  mutate(
    across(
      starts_with("mat") & where(is.character),
      ~ replace_na(parse_number(.x, na = c('', 'NA', '&')), 0)
    ),
    across(
      starts_with("mat") & where(is.numeric),
      ~ replace_na(.x, 0)
    ),
    ##Identificador de rezago
    rezago = if_else(
      ((tot_resid / num_cuarto) > 2.5) |
        (mat_pared %in% 1:6) |
        (mat_techos %in% c(1:4, 6, 7, 9)) |
        (mat_pisos == 1) |
        (excusado == valor_excusado),
      "En rezago",
      "Fuera de rezago"
    ),
    ##Clave de entidad
    cve_ent = case_when(
      nchar(folioviv) == 9  ~ paste0("0", substr(folioviv, 1, 1)),
      nchar(folioviv) == 10 ~ substr(folioviv, 1, 2)
    ),
    #Crear nombre de entidad
    nom_ent=case_when(
      cve_ent == "01" ~ "Aguascalientes",
      cve_ent == "02" ~ "Baja California",
      cve_ent == "03" ~ "Baja California Sur",
      cve_ent == "04" ~ "Campeche",
      cve_ent == "05" ~ "Chiapas",
      cve_ent == "06" ~ "Chihuahua",
      cve_ent == "07" ~ "Coahuila",
      cve_ent == "08" ~ "Colima",
      cve_ent == "09" ~ "Ciudad de México",
      cve_ent == "10" ~ "Durango",
      cve_ent == "11" ~ "Guanajuato",
      cve_ent == "12" ~ "Guerrero",
      cve_ent == "13" ~ "Hidalgo",
      cve_ent == "14" ~ "Jalisco",
      cve_ent == "15" ~ "México",
      cve_ent == "16" ~ "Michoacán",
      cve_ent == "17" ~ "Morelos",
      cve_ent == "18" ~ "Nayarit",
      cve_ent == "19" ~ "Nuevo León",
      cve_ent == "20" ~ "Oaxaca",
      cve_ent == "21" ~ "Puebla",
      cve_ent == "22" ~ "Querétaro Arteaga",
      cve_ent == "23" ~ "Quintana Roo",
      cve_ent == "24" ~ "San Luis Potosí",
      cve_ent == "25" ~ "Sinaloa",
      cve_ent == "26" ~ "Sonora",
      cve_ent == "27" ~ "Tabasco",
      cve_ent == "28" ~ "Tamaulipas",
      cve_ent == "29" ~ "Tlaxcala",
      cve_ent == "30" ~ 	"Veracruz",
      cve_ent == "31" ~ "Yucatán",
      cve_ent == "32" ~ "Zacatecas",
      TRUE 	~ NA_character_
    ))

Warning message:
“One or more parsing issues, call `problems()` on your data frame for details,
e.g.:
  dat <- vroom(...)
  problems(dat)”
Rows: 90324 Columns: 82
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (8): folioviv, mat_techos, finan_8, num_dueno1, num_dueno2, ubica_geo, ...
dbl (74): tipo_viv, mat_pared, mat_pisos, antiguedad, antigua_ne, cocina, co...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [33]:
##Crear diseño muestral

dm <- viviendas %>%
    as_survey_design(ids = upm, strata = est_dis, weights = factor)

In [34]:
##Calcular total de viviendas

dm%>%
  summarise(viviendas=survey_total(vartype=c("se","ci","cv")))

viviendas,viviendas_se,viviendas_low,viviendas_upp,viviendas_cv
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
38356042,142516.3,38076681,38635403,0.003715616


In [35]:
##Calcular rezago habitacional general

dm%>%
  group_by(rezago)%>%
  summarise(viviendas=survey_total(vartype=c("se","ci","cv")))

rezago,viviendas,viviendas_se,viviendas_low,viviendas_upp,viviendas_cv
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
En rezago,8381545,106629.1,8172530,8590560,0.01272189
Fuera de rezago,29974497,144400.9,29691442,30257552,0.00481746


In [36]:
##Calcular rezago habitacional por entidad federativa
dm%>%
  group_by(rezago, nom_ent)%>%
  summarise(viviendas=survey_total(vartype=c("se","ci","cv")))

rezago,nom_ent,viviendas,viviendas_se,viviendas_low,viviendas_upp,viviendas_cv
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
En rezago,Aguascalientes,16755,2076.157,12685.31,20824.69,0.12391269
En rezago,Baja California,474035,23282.208,428397.12,519672.88,0.04911496
En rezago,Baja California Sur,58987,3828.600,51482.16,66491.84,0.06490582
En rezago,Campeche,115596,4870.217,106049.38,125142.62,0.04213137
En rezago,Chiapas,88072,7561.688,73249.55,102894.45,0.08585803
En rezago,Chihuahua,54659,2865.524,49041.99,60276.01,0.05242548
En rezago,Ciudad de México,182009,17552.150,147603.21,216414.79,0.09643562
En rezago,Coahuila,1010514,40909.787,930322.48,1090705.52,0.04048414
En rezago,Colima,441301,18699.856,404645.47,477956.53,0.04237438
